In [1]:
import pandas as pd

df = pd.read_csv("telco.csv")

print(df.shape)

(7043, 50)


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Under 30                           7043 non-null   object 
 4   Senior Citizen                     7043 non-null   object 
 5   Married                            7043 non-null   object 
 6   Dependents                         7043 non-null   object 
 7   Number of Dependents               7043 non-null   int64  
 8   Country                            7043 non-null   object 
 9   State                              7043 non-null   object 
 10  City                               7043 non-null   object 
 11  Zip Code                           7043 non-null   int64

In [3]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)

Offer             3877
Internet Type     1526
Churn Category    5174
Churn Reason      5174
dtype: int64


In [4]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [5]:
financial_cols = [
    "Monthly Charge", "Total Charges", "Total Refunds",
    "Total Extra Data Charges", "Total Long Distance Charges"
]

for col in financial_cols:
    print(col, "negative values:", (df[col] < 0).sum())

Monthly Charge negative values: 0
Total Charges negative values: 0
Total Refunds negative values: 0
Total Extra Data Charges negative values: 0
Total Long Distance Charges negative values: 0


In [6]:
numerical_cols = [
    "Age", "Number of Dependents", "Number of Referrals", "Tenure in Months",
    "Monthly Charge", "Total Charges", "Total Refunds",
    "Total Extra Data Charges", "Total Long Distance Charges",
    "Population", "CLTV"
]

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = round((outliers / len(df)) * 100, 2)
    print(f"{col}: {outliers} outliers ({pct}%)")

Age: 0 outliers (0.0%)
Number of Dependents: 1627 outliers (23.1%)
Number of Referrals: 676 outliers (9.6%)
Tenure in Months: 0 outliers (0.0%)
Monthly Charge: 0 outliers (0.0%)
Total Charges: 0 outliers (0.0%)
Total Refunds: 525 outliers (7.45%)
Total Extra Data Charges: 728 outliers (10.34%)
Total Long Distance Charges: 196 outliers (2.78%)
Population: 57 outliers (0.81%)
CLTV: 0 outliers (0.0%)


In [7]:
print(df["Offer"].value_counts(dropna=False))

Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64


In [8]:
df["Offer"] = df["Offer"].fillna("No Offer")

print(df["Offer"].value_counts(dropna=False))

Offer
No Offer    3877
Offer B      824
Offer E      805
Offer D      602
Offer A      520
Offer C      415
Name: count, dtype: int64


In [9]:
print(df["Internet Type"].value_counts(dropna=False))

Internet Type
Fiber Optic    3035
DSL            1652
NaN            1526
Cable           830
Name: count, dtype: int64


In [10]:
print(df.groupby("Internet Service")["Internet Type"].apply(lambda x: x.isnull().sum()))

Internet Service
No     1526
Yes       0
Name: Internet Type, dtype: int64


In [11]:
df["Internet Type"] = df["Internet Type"].fillna("No Internet Service")

print(df["Internet Type"].value_counts(dropna=False))

Internet Type
Fiber Optic            3035
DSL                    1652
No Internet Service    1526
Cable                   830
Name: count, dtype: int64


In [12]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)

Churn Category    5174
Churn Reason      5174
dtype: int64


In [13]:
df.to_csv("cleaned_data.csv", index=False)
print("Saved successfully")

Saved successfully


## Data Cleaning & Data Quality — Member 1

### 1. Data Types
- Verified all 50 columns using `df.info()`.
- Total Charges was already stored as `float64` (numeric) — no conversion was needed.

### 2. Missing Values
- Found missing values in 4 columns: Offer (3877), Internet Type (1526), Churn Category (5174), Churn Reason (5174).
- **Offer**: Missingness represents customers who were not given any promotional offer. Filled with "No Offer".
- **Internet Type**: Confirmed via cross-check with Internet Service that all missing values belong to customers with no internet service. Filled with "No Internet Service".
- **Churn Category / Churn Reason**: Left as-is. These are only populated for churned customers and represent post-outcome information (would cause data leakage if used as predictors). To be excluded from features by Member 2.

### 3. Duplicates
- Checked using `df.duplicated().sum()`. Result: 0 duplicate rows. No action needed.

### 4. Invalid Values
- Checked all financial columns (Monthly Charge, Total Charges, Total Refunds, Total Extra Data Charges, Total Long Distance Charges) for negative values. Result: 0 negative values in all columns. No action needed.

### 5. Outliers (IQR method)
- Checked 11 numerical columns using the 1.5×IQR rule.
- Age, Tenure, Monthly Charge, Total Charges, CLTV: 0% outliers.
- Number of Dependents (23.1%), Number of Referrals (9.6%), Total Refunds (7.45%), Total Extra Data Charges (10.34%), Total Long Distance Charges (2.78%), Population (0.81%): flagged as statistical outliers.
- **Decision**: No outliers were removed or capped. These reflect natural right-skewed distributions (most customers at 0, fewer with higher values), not data errors. Removing them would discard legitimate customer information.

### 6. Output
- Cleaned dataset saved as `cleaned_data.csv` (7043 rows × 50 columns, same shape as original — only value-level fixes applied, no rows/columns dropped).